# QLoRA 학습 — 충남대 Q&A 시스템

Colab T4 GPU (15GB VRAM) 에서 실행합니다.

| 항목 | 값 |
|------|-----|
| Base | `Qwen/Qwen3-8B` (4bit NF4) |
| QLoRA | r=32, alpha=64, dropout=0.1, target=q/k/v/o/up/down_proj |
| 데이터 | HF Hub `adoveflash/cnu-qa-system` → `data/qa/train_clean.jsonl` |
| 학습 | 컨텍스트 포함 학습 + 답변 토큰에만 loss 적용 |
| 저장 | Google Drive + HF Hub 이중 백업 |

> **런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 전체 실행

## 1. 환경 설정

In [ ]:
%%time
!pip install -q torch transformers accelerate bitsandbytes peft datasets huggingface_hub

In [ ]:
import os
import json
import random
import torch

SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)

# ── 설정값 (여기만 수정) ──
MODEL_NAME = "Qwen/Qwen3-8B"
HF_REPO = "adoveflash/cnu-qa-system"
DRIVE_OUTPUT = "/content/drive/MyDrive/cnu_lora_adapter"
LOCAL_OUTPUT = "models/lora_adapter"
CKPT_DIR = f"{DRIVE_OUTPUT}/checkpoints"
MAX_LENGTH = 768
NUM_EPOCHS = 5
BATCH_SIZE = 1
GRAD_ACCUM = 16
LR = 2e-4

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2. HF Hub 로그인 & 데이터 다운로드

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f"Drive 저장 경로: {DRIVE_OUTPUT}")

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from huggingface_hub import snapshot_download

TRAIN_PATH = "data/qa/train_clean.jsonl"
EVAL_PATH = "data/qa/eval.jsonl"
CHUNKS_PATH = "data/corpus/chunks.jsonl"

# HF Hub에서 필요한 파일 다운로드
for path in [TRAIN_PATH, EVAL_PATH, CHUNKS_PATH]:
    if not os.path.exists(path):
        snapshot_download(
            repo_id=HF_REPO,
            local_dir=".",
            allow_patterns=[path],
        )

with open(TRAIN_PATH) as f:
    train_data = [json.loads(line) for line in f if line.strip()]

with open(EVAL_PATH) as f:
    eval_data = [json.loads(line) for line in f if line.strip()]

# 청크 매핑 로드 (chunk_id → text)
chunks_map = {}
with open(CHUNKS_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            chunk = json.loads(line)
            chunks_map[chunk["chunk_id"]] = chunk["text"]

print(f"학습: {len(train_data)}건, 검증: {len(eval_data)}건, 청크: {len(chunks_map)}개")

## 3. 모델 & 토크나이저 로드

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("[1/2] 토크나이저 로드...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("[2/2] 모델 로드 (4bit NF4)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model.gradient_checkpointing_enable()

vram_gb = torch.cuda.memory_reserved() / 1024**3
print(f"모델 로드 완료 — VRAM: {vram_gb:.2f} GB")

## 4. LoRA 설정 & 적용

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=32,
    lora_alpha=64,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. 데이터셋 준비

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = (
    "당신은 충남대학교 학내 정보 안내 도우미입니다.\n"
    "규칙:\n"
    "1. 반드시 주어진 참고 자료에 있는 정보만 사용하여 답변하세요.\n"
    "2. 참고 자료에 없는 내용은 추측하지 말고 '확인되지 않은 정보입니다'라고 답하세요.\n"
    "3. 답변은 간결하고 정확하게 작성하세요.\n"
    "4. 답변 끝에 출처 URL을 포함하세요."
)


def format_messages(qa, chunk_text=""):
    """추론 시와 동일한 형식으로 메시지를 구성한다."""
    if chunk_text:
        user_content = f"참고 자료:\n{chunk_text}\n\n질문: {qa['question']}"
    else:
        user_content = qa["question"]
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": qa["answer"]},
    ]


def find_assistant_start(messages):
    """assistant 응답 시작 토큰 위치를 찾는다."""
    prompt_only = messages[:2]
    prompt_text = tokenizer.apply_chat_template(
        prompt_only, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    return len(prompt_ids)


def build_dataset(data_list):
    """QA 리스트를 컨텍스트 포함 + loss masking Dataset으로 변환한다."""
    all_input_ids = []
    all_attention_mask = []
    all_labels = []

    for qa in data_list:
        chunk_text = chunks_map.get(qa.get("chunk_id", ""), "")
        messages = format_messages(qa, chunk_text)

        full_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False, enable_thinking=False
        )
        tokenized = tokenizer(
            full_text,
            truncation=True,
            max_length=MAX_LENGTH,
            padding="max_length",
            add_special_tokens=False,
        )

        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]

        # 답변 토큰에만 loss 적용
        assistant_start = find_assistant_start(messages)
        labels = [-100] * min(assistant_start, len(input_ids))
        labels += input_ids[len(labels):]

        # padding도 -100
        labels = [
            lb if am == 1 else -100
            for lb, am in zip(labels, attention_mask)
        ]
        if len(labels) < MAX_LENGTH:
            labels += [-100] * (MAX_LENGTH - len(labels))
        labels = labels[:MAX_LENGTH]

        all_input_ids.append(input_ids)
        all_attention_mask.append(attention_mask)
        all_labels.append(labels)

    return Dataset.from_dict({
        "input_ids": all_input_ids,
        "attention_mask": all_attention_mask,
        "labels": all_labels,
    })


random.seed(SEED)
random.shuffle(train_data)

train_dataset = build_dataset(train_data)
eval_dataset = build_dataset(eval_data)
print(f"학습 데이터셋: {len(train_dataset)}건")
print(f"검증 데이터셋: {len(eval_dataset)}건")

# 확인: loss가 걸리는 토큰 비율
sample_labels = train_dataset[0]["labels"]
loss_tokens = sum(1 for l in sample_labels if l != -100)
total_tokens = sum(1 for a in train_dataset[0]["attention_mask"] if a == 1)
print(f"샘플 — 전체 토큰: {total_tokens}, loss 토큰: {loss_tokens} ({loss_tokens/max(total_tokens,1)*100:.1f}%)")

## 6. 학습

In [ ]:
from transformers import TrainingArguments, Trainer, TrainerCallback
from huggingface_hub import HfApi


class DriveBackupCallback(TrainerCallback):
    """에폭 종료 시 HF Hub에 백업한다."""
    def on_save(self, args, state, control, **kwargs):
        epoch = int(state.epoch) if state.epoch else 0
        print(f"\n에폭 {epoch} → HF Hub 백업 중...")
        try:
            api = HfApi()
            ckpts = sorted(
                [d for d in os.listdir(CKPT_DIR) if d.startswith("checkpoint-")],
                key=lambda x: int(x.split("-")[1]),
            )
            if ckpts:
                latest_path = os.path.join(CKPT_DIR, ckpts[-1])
                api.upload_folder(
                    folder_path=latest_path,
                    path_in_repo="models/lora_adapter",
                    repo_id=HF_REPO,
                )
                print(f"  백업 완료: {ckpts[-1]}")
        except Exception as e:
            print(f"  백업 실패 (학습은 계속): {e}")


training_args = TrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.1,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    seed=SEED,
    fp16=True,
    report_to="none",
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    callbacks=[DriveBackupCallback()],
)

# 체크포인트에서 이어서 학습
resume_ckpt = None
if os.path.exists(CKPT_DIR):
    ckpts = [d for d in os.listdir(CKPT_DIR) if d.startswith("checkpoint-")]
    if ckpts:
        resume_ckpt = os.path.join(
            CKPT_DIR,
            sorted(ckpts, key=lambda x: int(x.split("-")[1]))[-1],
        )
        print(f"체크포인트에서 재개: {resume_ckpt}")

print("학습 시작!")
trainer.train(resume_from_checkpoint=resume_ckpt)
print("학습 완료!")

## 7. 학습 결과 확인

In [ ]:
# 학습 로그 출력
for log in trainer.state.log_history:
    if "eval_loss" in log:
        print(f"에폭 {log.get('epoch', '?'):.0f} — train_loss: {log.get('loss', 'N/A')}, eval_loss: {log['eval_loss']:.4f}")

In [ ]:
# 어댑터 저장 (로컬 + Drive)
os.makedirs(LOCAL_OUTPUT, exist_ok=True)
model.save_pretrained(LOCAL_OUTPUT)
tokenizer.save_pretrained(LOCAL_OUTPUT)
print(f"로컬 저장 완료: {LOCAL_OUTPUT}")

# Drive에도 복사
import shutil
drive_adapter = f"{DRIVE_OUTPUT}/final_adapter"
if os.path.exists(drive_adapter):
    shutil.rmtree(drive_adapter)
shutil.copytree(LOCAL_OUTPUT, drive_adapter)
print(f"Drive 저장 완료: {drive_adapter}")

# 파일 목록
for f in os.listdir(LOCAL_OUTPUT):
    size = os.path.getsize(os.path.join(LOCAL_OUTPUT, f))
    print(f"  {f}: {size / 1024**2:.1f} MB")

## 8. HF Hub 업로드

In [ ]:
api = HfApi()
api.upload_folder(
    folder_path=LOCAL_OUTPUT,
    path_in_repo="models/lora_adapter",
    repo_id=HF_REPO,
)
print(f"HF Hub 업로드 완료: {HF_REPO}/models/lora_adapter")

## 9. 추론 테스트

In [ ]:
import re

model.eval()
model.config.use_cache = True  # 추론 시 KV cache 활성화

_THINK_RE = re.compile(r"<think>.*?</think>\s*", flags=re.DOTALL)

test_questions = [
    "컴퓨터융합학부 졸업 요건이 어떻게 되나요?",
    "수강신청은 언제 하나요?",
    "오늘 학식 뭐 나와요?",
    "셔틀버스 시간표 알려주세요",
]

for q in test_questions:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": q},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    torch.manual_seed(SEED)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=192,
            do_sample=False,
            repetition_penalty=1.2,
        )
    gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    answer = _THINK_RE.sub("", answer).strip()
    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 60)

## 10. 벡터 DB 재구축

bge-m3로 청크를 임베딩하고 ChromaDB 인덱스를 재구축합니다.  
구축 완료 후 HF Hub에 업로드하여 로컬에서 다운로드할 수 있도록 합니다.

In [ ]:
!pip install -q sentence-transformers chromadb

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

# 학습에 사용한 LLM 메모리 해제 (임베딩 모델 로드를 위해)
import gc
del model, trainer
gc.collect()
torch.cuda.empty_cache()
print(f"메모리 해제 후 VRAM: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

# 임베딩 모델 로드
print("임베딩 모델 로드: BAAI/bge-m3")
embed_model = SentenceTransformer("BAAI/bge-m3")

# 청크 로드
chunks = []
with open(CHUNKS_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            chunks.append(json.loads(line))
print(f"청크 수: {len(chunks)}")

# 임베딩 생성
texts = [c["text"] for c in chunks]
print(f"{len(texts)}개 청크 임베딩 중...")
embeddings = embed_model.encode(texts, batch_size=32, show_progress_bar=True)
print(f"임베딩 완료: shape={embeddings.shape}")

# ChromaDB 인덱스 구축
DB_PATH = "data/vector_db"
os.makedirs(DB_PATH, exist_ok=True)
client = chromadb.PersistentClient(path=DB_PATH)

# 기존 컬렉션 삭제 후 재생성
try:
    client.delete_collection("cnu_chunks")
except Exception:
    pass

collection = client.create_collection(
    name="cnu_chunks",
    metadata={"hnsw:space": "cosine"},
)

# 배치로 추가 (ChromaDB 제한)
BATCH = 500
for i in range(0, len(chunks), BATCH):
    batch_chunks = chunks[i:i + BATCH]
    batch_embeds = embeddings[i:i + BATCH].tolist()
    collection.add(
        ids=[c["chunk_id"] for c in batch_chunks],
        embeddings=batch_embeds,
        documents=[c["text"] for c in batch_chunks],
        metadatas=[{"url": c["url"], "title": c["title"], "source": c["source"]} for c in batch_chunks],
    )

print(f"벡터 DB 구축 완료: {collection.count()}개 인덱싱")

# 검색 테스트
test_q = "졸업 요건이 어떻게 되나요?"
q_emb = embed_model.encode([test_q]).tolist()[0]
results = collection.query(query_embeddings=[q_emb], n_results=3, include=["documents"])
print(f"\n검색 테스트: '{test_q}'")
for i, doc in enumerate(results["documents"][0]):
    print(f"  [{i+1}] {doc[:80]}...")

In [ ]:
# 벡터 DB를 HF Hub에 업로드
api = HfApi()
api.upload_folder(
    folder_path=DB_PATH,
    path_in_repo="data/vector_db",
    repo_id=HF_REPO,
)
print(f"벡터 DB 업로드 완료: {HF_REPO}/data/vector_db")

# Drive에도 백업
import shutil
drive_db = f"{DRIVE_OUTPUT}/vector_db"
if os.path.exists(drive_db):
    shutil.rmtree(drive_db)
shutil.copytree(DB_PATH, drive_db)
print(f"Drive 백업 완료: {drive_db}")